In [ ]:
# Updated for Spring 2026 EE300005

# Import all libraries

# TensorFlow is an open source machine learning library
import tensorflow as tf
import keras
# NumPy is a math library
import numpy as np
# Matplotlib is a graphing library
import matplotlib.pyplot as plt
# math is Python's math library
import math

In [ ]:
# We'll generate this many sample datapoints
SAMPLES = 1000

# Set a "seed" value, so we get the same random numbers each time we run this # notebook. Any number can be used here.
np.random.seed(1337)

# Generate a uniformly distributed set of random numbers in the range from # 0 to 2π, which covers a complete sine wave oscillation
x_values = np.random.uniform(low=0, high=2*math.pi, size=SAMPLES)

# Shuffle the values to guarantee they're not in order
np.random.shuffle(x_values)

# Calculate the corresponding function values
y_sine     = np.sin(3 * x_values)
y_rect     = np.abs(np.sin(1.5 * x_values))
y_sawtooth = 2 * (3*x_values/(2*np.pi) - np.floor(3*x_values/(2*np.pi) + 0.5))

# Plot our data. The 'b.' argument tells the library to print blue dots.
plt.figure(figsize=(10, 5))
plt.plot(x_values, y_sine,     'b.', label='Sine (3x)',          alpha=0.5)
plt.plot(x_values, y_rect,     'r.', label='Rectified Sine (1.5x)', alpha=0.5)
plt.plot(x_values, y_sawtooth, 'g.', label='Sawtooth (3x)',      alpha=0.5)
plt.legend()
plt.title('Task 1.1: Training Data Visualization')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
# Add some noises to function values
y_sine += 0.1 * np.random.randn(*y_sine.shape)
y_rect += 0.1 * np.random.randn(*y_rect.shape)
y_sawtooth += 0.1 * np.random.randn(*y_sawtooth.shape)

# Plot our data. The 'b.' argument tells the library to print blue dots.
plt.figure(figsize=(10, 5))
plt.plot(x_values, y_sine,     'b.', label='Sine (3x)',          alpha=0.5)
plt.plot(x_values, y_rect,     'r.', label='Rectified Sine (1.5x)', alpha=0.5)
plt.plot(x_values, y_sawtooth, 'g.', label='Sawtooth (3x)',      alpha=0.5)
plt.legend()
plt.title('Task 1.1: Training Data Visualization')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

In [ ]:
# We'll use 60% of our data for training and 20% for testing.
# The remaining 20% # will be used for validation.
# Calculate the indices of each section.
TRAIN_SPLIT = int(0.6 * SAMPLES)
TEST_SPLIT = int(0.2 * SAMPLES + TRAIN_SPLIT)

# Use np.split to chop our data into three parts.
# The second argument to np.split is an array of indices where the data will be
# split. We provide two indices, so the data will be divided into three chunks.
x_train, x_validate, x_test = np.split(x_values, [TRAIN_SPLIT, TEST_SPLIT])

# Sine
y_sine_train, y_sine_validate, y_sine_test = np.split(y_sine, [TRAIN_SPLIT, TEST_SPLIT])
# Rectified Sine
y_rect_train, y_rect_validate, y_rect_test = np.split(y_rect, [TRAIN_SPLIT, TEST_SPLIT])
# Sawtooth
y_sawtooth_train, y_sawtooth_validate, y_sawtooth_test = np.split(y_sawtooth, [TRAIN_SPLIT, TEST_SPLIT])

# Double check that our splits add up correctly
assert (x_train.size + x_validate.size + x_test.size) == SAMPLES

assert (y_sine_train.size + y_sine_validate.size + y_sine_test.size) == SAMPLES
assert (y_rect_train.size + y_rect_validate.size + y_rect_test.size) == SAMPLES
assert (y_sawtooth_train.size + y_sawtooth_validate.size + y_sawtooth_test.size) == SAMPLES

# Plot the data in each partition in different colors:
plt.plot(x_train, y_sine_train, 'b.', label="Train")
plt.plot(x_validate, y_sine_validate, 'y.', label="Validate")
plt.plot(x_test, y_sine_test, 'r.', label="Test")
plt.legend()
plt.show()

In [ ]:
# We'll use Keras to create a simple model architecture
from keras import layers
model_1 = keras.Sequential()

# First layer takes a scalar input and feeds it through 16 "neurons." The
# neurons decide whether to activate based on the 'relu' activation function.
model_1.add(layers.Dense(16, activation='relu', input_shape=(1,)))

# Final layer is a single neuron, since we want to output a single value
model_1.add(layers.Dense(1))
# Compile the model using a standard optimizer and loss function for regression
model_1.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

# Print a summary of the model's architecture
model_1.summary()

In [ ]:
history_1 = model_1.fit(x_train, y_sine_train, epochs=1000, batch_size=16,
                         validation_data=(x_validate, y_sine_validate))

In [ ]:
loss = history_1.history['loss']
val_loss = history_1.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'g.', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
SKIP = 100
plt.plot(epochs[SKIP:], loss[SKIP:], 'g.', label='Training loss')
plt.plot(epochs[SKIP:], val_loss[SKIP:], 'b.', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
# Draw a graph of mean absolute error, which is another way of # measuring the amount of error in the prediction.
mae = history_1.history['mae']
val_mae = history_1.history['val_mae']

plt.plot(epochs[SKIP:], mae[SKIP:], 'g.', label='Training MAE')
plt.plot(epochs[SKIP:], val_mae[SKIP:], 'b.', label='Validation MAE')
plt.title('Training and validation mean absolute error')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.show()

In [ ]:
# Use the model to make predictions from our validation data
predictions = model_1.predict(x_train)

# Plot the predictions along with the test data
plt.clf()
plt.title('Training data predicted vs actual values')
plt.plot(x_test, y_sine_test, 'b.', label='Actual')
plt.plot(x_train, predictions, 'r.', label='Predicted')
plt.legend()
plt.show()

In [ ]:
def build_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(64, activation='relu', input_shape=(1,)))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(1))
    model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
    return model

model_sine = build_model()
model_rect = build_model()
model_saw  = build_model()

model_sine.summary()


In [ ]:
EPOCHS = 600

history_sine = model_sine.fit(
    x_train, y_sine_train, epochs=EPOCHS, batch_size=16,
    validation_data=(x_validate, y_sine_validate))

In [ ]:
history_rect = model_rect.fit(
    x_train, y_rect_train, epochs=EPOCHS, batch_size=16,
    validation_data=(x_validate, y_rect_validate))

In [ ]:
history_saw = model_saw.fit(
    x_train, y_sawtooth_train, epochs=EPOCHS, batch_size=16,
    validation_data=(x_validate, y_sawtooth_validate))

In [ ]:
SKIP = 100

fig, axes = plt.subplots(3, 2, figsize=(12, 12))

datasets = [
    (history_sine, 'Sine'),
    (history_rect, 'Rectified Sine'),
    (history_saw,  'Sawtooth')
]

for row, (history, name) in enumerate(datasets):
    epochs = range(1, len(history.history['loss']) + 1)

    # 왼쪽: Loss
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    axes[row][0].plot(list(epochs)[SKIP:], loss[SKIP:],     'g.', label='Train loss')
    axes[row][0].plot(list(epochs)[SKIP:], val_loss[SKIP:], 'b.', label='Val loss')
    axes[row][0].set_title(f'{name} - Loss (from epoch {SKIP})')
    axes[row][0].set_xlabel('Epochs')
    axes[row][0].set_ylabel('Loss (MSE)')
    axes[row][0].legend()

    # 오른쪽: MAE
    mae     = history.history['mae']
    val_mae = history.history['val_mae']
    axes[row][1].plot(list(epochs)[SKIP:], mae[SKIP:],     'g.', label='Train MAE')
    axes[row][1].plot(list(epochs)[SKIP:], val_mae[SKIP:], 'b.', label='Val MAE')
    axes[row][1].set_title(f'{name} - MAE (from epoch {SKIP})')
    axes[row][1].set_xlabel('Epochs')
    axes[row][1].set_ylabel('MAE')
    axes[row][1].legend()

plt.tight_layout()
plt.suptitle('Task 1.2: Training and Validation Loss/MAE', y=1.02)
plt.show()

In [ ]:
# 평가
print("=== Test Set Evaluation ===")
loss_sine, mae_sine = model_sine.evaluate(x_test, y_sine_test)
loss_rect, mae_rect = model_rect.evaluate(x_test, y_rect_test)
loss_saw,  mae_saw  = model_saw.evaluate(x_test,  y_sawtooth_test)

print(f"Sine     - Loss: {loss_sine:.4f}, MAE: {mae_sine:.4f}")
print(f"Rect     - Loss: {loss_rect:.4f}, MAE: {mae_rect:.4f}")
print(f"Sawtooth - Loss: {loss_saw:.4f},  MAE: {mae_saw:.4f}")

# 플롯
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model, x_t, y_t, name) in zip(axes, [
    (model_sine, x_test, y_sine_test, 'Sine'),
    (model_rect, x_test, y_rect_test, 'Rectified Sine'),
    (model_saw,  x_test, y_sawtooth_test,  'Sawtooth')
]):
    preds = model.predict(x_t, verbose=0)
    ax.plot(x_t, y_t,    'b.', label='Actual',    alpha=0.5)
    ax.plot(x_t, preds,  'r.', label='Predicted', alpha=0.5)
    ax.set_title(name)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
def convert_float(model, name):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    filename = f'{name}_float.tflite'
    with open(filename, 'wb') as f:
        f.write(tflite_model)
    print(f"Saved {filename}: {len(tflite_model)} bytes")
    return tflite_model

sine_float = convert_float(model_sine, 'sine')
rect_float = convert_float(model_rect, 'rect')
saw_float  = convert_float(model_saw,  'sawtooth')

In [ ]:
def convert_quant(model, x_train_data, name):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = tf.int8
    converter.inference_output_type = tf.int8

    def rep_dataset():
        for val in x_train_data:
            yield [np.array(val, dtype=np.float32, ndmin=2)]

    converter.representative_dataset = rep_dataset
    tflite_model = converter.convert()
    filename = f'{name}_quant.tflite'
    with open(filename, 'wb') as f:
        f.write(tflite_model)
    print(f"Saved {filename}: {len(tflite_model)} bytes")
    return tflite_model

sine_quant = convert_quant(model_sine, x_train, 'sine')
rect_quant = convert_quant(model_rect, x_train, 'rect')
saw_quant  = convert_quant(model_saw,  x_train, 'sawtooth')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 15))

datasets = [
    ('sine',     'sine_float.tflite',     'sine_quant.tflite',
     x_test, y_sine_test,  'Sine'),
    ('rect',     'rect_float.tflite',     'rect_quant.tflite',
     x_test, y_rect_test,  'Rectified Sine'),
    ('sawtooth', 'sawtooth_float.tflite', 'sawtooth_quant.tflite',
     x_test, y_sawtooth_test,   'Sawtooth'),
]

for ax, (name, float_path, quant_path, x_t, y_t, title) in zip(axes, datasets):

    # --- Float32 모델 로드 ---
    float_interp = tf.lite.Interpreter(float_path)
    float_interp.allocate_tensors()
    float_in_idx  = float_interp.get_input_details()[0]['index']
    float_out_idx = float_interp.get_output_details()[0]['index']

    # --- Quant 모델 로드 ---
    quant_interp = tf.lite.Interpreter(quant_path)
    quant_interp.allocate_tensors()
    quant_in_idx  = quant_interp.get_input_details()[0]['index']
    quant_out_idx = quant_interp.get_output_details()[0]['index']
    in_scale,  in_zero  = quant_interp.get_input_details()[0]['quantization']
    out_scale, out_zero = quant_interp.get_output_details()[0]['quantization']

    float_preds = []
    quant_preds = []

    for x_value in x_t:
        # Float32 추론
        x_tensor = tf.convert_to_tensor([[x_value]], dtype=np.float32)
        float_interp.set_tensor(float_in_idx, x_tensor)
        float_interp.invoke()
        float_preds.append(float_interp.get_tensor(float_out_idx)[0])

        # INT8 추론
        x_quant = np.array([[int(x_value / in_scale + in_zero)]], dtype=np.int8)
        quant_interp.set_tensor(quant_in_idx, x_quant)
        quant_interp.invoke()
        y_q = quant_interp.get_tensor(quant_out_idx)[0][0]
        quant_preds.append((float(y_q) - out_zero) * out_scale)

    ax.plot(x_t, y_t,          'b.', label='Actual',       alpha=0.5)
    ax.plot(x_t, float_preds,  'g.', label='Float32 Lite', alpha=0.7)
    ax.plot(x_t, quant_preds,  'r.', label='INT8 Quant',   alpha=0.7)
    ax.set_title(f'{title} - Predicted vs Actual')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
model_files = [
    ('sine_float.tflite',     'sine_quant.tflite',     'Sine'),
    ('rect_float.tflite',     'rect_quant.tflite',     'Rectified Sine'),
    ('sawtooth_float.tflite', 'sawtooth_quant.tflite', 'Sawtooth'),
]

for float_path, quant_path, name in model_files:
    with open(float_path, 'rb') as f:
        float_model = f.read()
    with open(quant_path, 'rb') as f:
        quant_model = f.read()

    print(f"=== {name} ===")
    print(f"Float32 size: {len(float_model)} bytes")
    print(f"INT8 size:    {len(quant_model)} bytes")
    print(f"Difference:   {len(quant_model) - len(float_model)} bytes")
    print(f"Ratio:        {len(quant_model) / len(float_model):.2%}")
    print()

In [ ]:
# Task 2.1: 6개 TFLite 모델 Loss/MAE 출력

datasets = {
    'sine':     (x_test, y_sine_test),
    'rect':     (x_test, y_rect_test),
    'sawtooth': (x_test, y_sawtooth_test),
}

models_info = [
    ('sine_float.tflite',     'sine',     False, 'Sine Float32'),
    ('sine_quant.tflite',     'sine',     True,  'Sine INT8'),
    ('rect_float.tflite',     'rect',     False, 'Rect Float32'),
    ('rect_quant.tflite',     'rect',     True,  'Rect INT8'),
    ('sawtooth_float.tflite', 'sawtooth', False, 'Sawtooth Float32'),
    ('sawtooth_quant.tflite', 'sawtooth', True,  'Sawtooth INT8'),
]

print(f"{'Model':<22} {'Loss (MSE)':>12} {'MAE':>10}")
print("-" * 46)

for path, func_name, is_quant, label in models_info:
    x_t, y_t = datasets[func_name]

    interp = tf.lite.Interpreter(path)
    interp.allocate_tensors()
    in_idx  = interp.get_input_details()[0]['index']
    out_idx = interp.get_output_details()[0]['index']

    if is_quant:
        in_scale,  in_zero  = interp.get_input_details()[0]['quantization']
        out_scale, out_zero = interp.get_output_details()[0]['quantization']

    preds = []
    for x in x_t:
        if is_quant:
            x_in = np.array([[int(x / in_scale + in_zero)]], dtype=np.int8)
        else:
            x_in = np.array([[x]], dtype=np.float32)
        interp.set_tensor(in_idx, x_in)
        interp.invoke()
        y = interp.get_tensor(out_idx)[0][0]
        if is_quant:
            y = (float(y) - out_zero) * out_scale
        preds.append(y)

    preds = np.array(preds)
    mse   = np.mean((preds - y_t) ** 2)
    mae   = np.mean(np.abs(preds - y_t))
    print(f"{label:<22} {mse:>12.6f} {mae:>10.6f}")

In [ ]:
# Instantiate an interpreter
sine_model = tf.lite.Interpreter('sine_model.tflite')
sine_model_quant = tf.lite.Interpreter('sine_model_quant.tflite')

# load the parameters to convert to and from the quantized data type
input_scale, input_zero_point = sine_model_quant.get_input_details()[0]["quantization"]
output_scale, output_zero_point = sine_model_quant.get_output_details()[0]["quantization"]

# Allocate memory for each model
sine_model.allocate_tensors()
sine_model_quant.allocate_tensors()

# Get indexes of the input and output tensors
sine_model_input_index  = sine_model.get_input_details() [0]["index"]
sine_model_output_index = sine_model.get_output_details()[0]["index"]

sine_model_quant_input_index  = sine_model_quant.get_input_details() [0]["index"]
sine_model_quant_output_index = sine_model_quant.get_output_details()[0]["index"]

# Create array to store the results
sine_model_predictions = []
sine_model_quant_predictions = []

# Run each model's interpreter for each value and store the results in arrays
for x_value in x_test:
  # Create a 2D tensor wrapping the current x value
  x_value_tensor = tf.convert_to_tensor([[x_value]], dtype=np.float32)
  x_value_quant = x_value / input_scale + input_zero_point
  x_value_quant = x_value_quant.astype(np.int8)
  x_value_tensorInt8 = tf.convert_to_tensor([[x_value_quant]], dtype=np.int8)

  # Write the value to the input tensor
  sine_model.set_tensor(sine_model_input_index, x_value_tensor)
  # Run inference
  sine_model.invoke()
  # Read the prediction from the output tensor
  sine_model_predictions.append(sine_model.get_tensor(sine_model_output_index)[0])

  # Ditto for the quant model
  sine_model_quant.set_tensor(sine_model_quant_input_index, x_value_tensorInt8)
  sine_model_quant.invoke()
  sine_model_quant_pred = sine_model_quant.get_tensor(sine_model_quant_output_index)[0]
  sine_model_quant_pred = sine_model_quant_pred.astype(np.float32)
  sine_model_quant_pred = (sine_model_quant_pred - output_zero_point) * output_scale
  sine_model_quant_predictions.append(sine_model_quant_pred)

# See how they line up with the data
plt.clf()
plt.title('Comparison of various models against actual values')
plt.plot(x_test, y_test, 'bo', label='Actual')
plt.plot(x_test, predictions, 'ro', label='Original predictions')
plt.plot(x_test, sine_model_predictions, 'gx', label='Lite predictions')
plt.plot(x_test, sine_model_quant_predictions, 'rx', label='LiteQuant predictions')
plt.legend()
plt.show()

In [ ]:
# Raspberry Pi Script - Copy this, uncomment it and run it on your Pi

# import math
# import numpy as np
# import plotext as plt
# from tflite_runtime.interpreter import Interpreter
# from pathlib import Path
# import time

# path = "sine_model_quant.tflite"

# if not Path(path).exists():
#     raise FileNotFoundError(f"Model not found: {path}")

# model_size = Path(path).stat().st_size
# print(f"Model size: {model_size} bytes ({model_size/1024:.2f} KB)")

# interp = Interpreter(model_path=path)
# interp.allocate_tensors()
# in_det, out_det = interp.get_input_details(), interp.get_output_details()

# # Get quantization parameters
# in_scale, in_zero = in_det[0]['quantization']
# out_scale, out_zero = out_det[0]['quantization']

# # Warmup
# for _ in range(10):
#     x_in = np.array([[int(0/in_scale + in_zero)]], dtype=np.int8)
#     interp.set_tensor(in_det[0]['index'], x_in)
#     interp.invoke()

# # Prepare test data
# x_vals = np.arange(0, 2*math.pi, 0.05)
# y_true = np.sin(x_vals)
# y_pred = []
# times = []

# # Benchmark individual inferences
# for x in x_vals:
#     x_in = np.array([[int(x/in_scale + in_zero)]], dtype=np.int8)
#     interp.set_tensor(in_det[0]['index'], x_in)

#     start = time.perf_counter()
#     interp.invoke()
#     end = time.perf_counter()

#     y = interp.get_tensor(out_det[0]['index'])[0][0]
#     y_pred.append((y - out_zero) * out_scale)
#     times.append((end - start) * 1000)

# y_pred = np.array(y_pred)
# times = np.array(times)

# # Performance metrics
# print(f"\n=== Timing Statistics ===")
# print(f"  Mean: {np.mean(times):.4f} ms (typical latency)")
# print(f"  Median: {np.median(times):.4f} ms (middle value)")
# print(f"  Std: {np.std(times):.4f} ms (consistency)")
# print(f"  Min: {np.min(times):.4f} ms (best case)")
# print(f"  Max: {np.max(times):.4f} ms (worst case)")
# print(f"  Inferences/sec: {1000/np.mean(times):.0f} (throughput)")

# # Accuracy metrics
# mae = np.mean(np.abs(y_pred - y_true))
# mse = np.mean((y_pred - y_true)**2)
# rmse = np.sqrt(mse)
# max_error = np.max(np.abs(y_pred - y_true))

# print(f"\n=== Accuracy Metrics ===")
# print(f"  MAE: {mae:.6f} (avg absolute error)")
# print(f"  RMSE: {rmse:.6f} (penalizes large errors)")
# print(f"  Max Error: {max_error:.6f} (worst case error)")

# print(f"\n=== Design Tradeoffs ===")
# print(f"  Speed: {1000/np.mean(times):.0f} inferences/sec")
# print(f"  Size: {model_size/1024:.1f} KB")
# print(f"  Accuracy: MAE={mae:.6f}")

# # Plot comparison
# plt.plot(x_vals, y_pred, label="Quantized Model")
# plt.plot(x_vals, y_true, label="True Sine")
# plt.title("Quantized Model vs True Sine")
# plt.xlabel("Input (radians)")
# plt.ylabel("Output")
# plt.show()